# IPL Evolution Data Analysis

This notebook explores how IPL cricket changed from 2008 to 2025 using the
summary datasets committed in this repository:

- `data/ipl_matches.csv`
- `data/ipl_batting_stats.csv`
- `data/ipl_bowling_stats.csv`

The raw Cricsheet ball-by-ball archive is intentionally not committed because
it is large. The optional `scripts/process_data.py` pipeline can rebuild these
summary files when `data/ipl_raw/` is available locally.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_DIR = Path("data")
matches = pd.read_csv(DATA_DIR / "ipl_matches.csv", parse_dates=["date"])
batting = pd.read_csv(DATA_DIR / "ipl_batting_stats.csv")
bowling = pd.read_csv(DATA_DIR / "ipl_bowling_stats.csv")

for frame in (matches, batting, bowling):
    frame["season"] = frame["season"].astype(str)

matches["season_int"] = matches["season"].astype(int)
batting["season_int"] = batting["season"].astype(int)
bowling["season_int"] = bowling["season"].astype(int)

print("Dataset summary")
print(f"Matches: {len(matches):,}")
print(f"Seasons: {matches['season'].min()}-{matches['season'].max()} ({matches['season'].nunique()} seasons)")
print(f"Batting season-player rows: {len(batting):,}")
print(f"Bowling season-player rows: {len(bowling):,}")

## 1. Scoring Has Accelerated

The first question is whether IPL scoring has increased over time. Match-level
data is enough to compare average match totals, run rate, and boundary volume
by season.

In [ ]:
season_scoring = (
    matches.groupby(["season", "season_int"], as_index=False)
    .agg(
        matches=("match_id", "count"),
        avg_total_runs=("total_runs", "mean"),
        avg_run_rate=("total_runs", lambda s: (s.sum() / matches.loc[s.index, "total_balls"].sum()) * 6),
        avg_balls_per_match=("total_balls", "mean"),
        fours_per_match=("total_fours", "mean"),
        sixes_per_match=("total_sixes", "mean"),
    )
    .sort_values("season_int")
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(
    x=season_scoring["season"],
    y=season_scoring["avg_total_runs"],
    name="Average match runs",
    marker_color="#10b981",
)
fig.add_scatter(
    x=season_scoring["season"],
    y=season_scoring["avg_run_rate"],
    name="Run rate",
    mode="lines+markers",
    line=dict(color="#f59e0b", width=3),
    secondary_y=True,
)
fig.update_layout(
    title="IPL scoring trend by season",
    template="plotly_dark",
    height=520,
    xaxis_title="Season",
)
fig.update_yaxes(title_text="Average match runs", secondary_y=False)
fig.update_yaxes(title_text="Runs per over", secondary_y=True)
fig.show()

season_scoring.tail(5)

## 2. Boundary Hitting Changed the Shape of Scores

Boundary counts show whether the scoring rise is driven by strike rotation,
fours, or six-hitting.

In [ ]:
boundary = season_scoring.assign(
    fours_per_100_balls=lambda df: (df["fours_per_match"] / (df["avg_balls_per_match"] / 100)).round(2),
    sixes_per_100_balls=lambda df: (df["sixes_per_match"] / (df["avg_balls_per_match"] / 100)).round(2),
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=boundary["season"],
    y=boundary["fours_per_match"],
    name="Fours per match",
    fill="tozeroy",
    line=dict(color="#06b6d4"),
))
fig.add_trace(go.Scatter(
    x=boundary["season"],
    y=boundary["sixes_per_match"],
    name="Sixes per match",
    fill="tozeroy",
    line=dict(color="#f59e0b"),
))
fig.update_layout(
    title="Boundary volume by season",
    template="plotly_dark",
    height=500,
    xaxis_title="Season",
    yaxis_title="Boundaries per match",
)
fig.show()

## 3. Bowlers Adapted Under Pressure

The bowler summary data lets us compare seasonal economy and wicket-taking.
The important signal is not just that economy increased, but whether wickets
collapsed at the same time.

In [ ]:
bowling_trend = (
    bowling.replace([float("inf")], pd.NA)
    .groupby(["season", "season_int"], as_index=False)
    .agg(
        avg_economy=("economy", "mean"),
        total_wickets=("wickets", "sum"),
        avg_bowling_sr=("bowling_sr", "mean"),
    )
    .sort_values("season_int")
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(
    x=bowling_trend["season"],
    y=bowling_trend["avg_economy"],
    mode="lines+markers",
    name="Average economy",
    line=dict(color="#ef4444", width=3),
)
fig.add_bar(
    x=bowling_trend["season"],
    y=bowling_trend["total_wickets"],
    name="Total wickets",
    marker_color="#8b5cf6",
    opacity=0.65,
    secondary_y=True,
)
fig.update_layout(
    title="Bowling pressure across IPL seasons",
    template="plotly_dark",
    height=520,
    xaxis_title="Season",
)
fig.update_yaxes(title_text="Economy", secondary_y=False)
fig.update_yaxes(title_text="Wickets", secondary_y=True)
fig.show()

## 4. Toss Advantage Is Limited

The match summary file tracks toss winners and match winners, so we can test
whether toss wins translate into match wins.

In [ ]:
toss = matches.assign(toss_won_match=matches["toss_winner"] == matches["winner"])
toss_trend = (
    toss.groupby(["season", "season_int"], as_index=False)
    .agg(
        toss_win_match_pct=("toss_won_match", "mean"),
        field_first_pct=("toss_decision", lambda s: (s == "field").mean()),
    )
    .sort_values("season_int")
)
toss_trend[["toss_win_match_pct", "field_first_pct"]] *= 100

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=toss_trend["season"],
    y=toss_trend["toss_win_match_pct"],
    mode="lines+markers",
    name="Toss winner also won match",
    line=dict(color="#10b981", width=3),
))
fig.add_trace(go.Scatter(
    x=toss_trend["season"],
    y=toss_trend["field_first_pct"],
    mode="lines+markers",
    name="Chose to field first",
    line=dict(color="#f59e0b", width=3),
))
fig.add_hline(y=50, line_dash="dash", line_color="#94a3b8")
fig.update_layout(
    title="Toss impact and chase preference",
    template="plotly_dark",
    height=520,
    xaxis_title="Season",
    yaxis_title="Percent",
)
fig.show()

## 5. Player Leaderboards

The summary tables are also useful for portfolio-style leaderboards: aggregate
player seasons into career totals, then compare volume and efficiency.

In [ ]:
batters = (
    batting.groupby("striker", as_index=False)
    .agg(
        runs=("runs", "sum"),
        balls=("balls_faced", "sum"),
        fours=("fours", "sum"),
        sixes=("sixes", "sum"),
        dismissals=("dismissals", "sum"),
    )
)
batters["strike_rate"] = (batters["runs"] / batters["balls"] * 100).round(2)
batters["average"] = (batters["runs"] / batters["dismissals"].replace(0, pd.NA)).round(2)
top_batters = batters.sort_values("runs", ascending=False).head(15)

fig = px.bar(
    top_batters,
    x="runs",
    y="striker",
    color="strike_rate",
    orientation="h",
    title="Top IPL run scorers in the included dataset",
    template="plotly_dark",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=620, yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
bowlers = (
    bowling.groupby("bowler", as_index=False)
    .agg(
        wickets=("wickets", "sum"),
        runs_conceded=("runs_conceded", "sum"),
        extras=("extras_conceded", "sum"),
        balls=("balls_bowled", "sum"),
    )
)
bowlers["economy"] = ((bowlers["runs_conceded"] + bowlers["extras"]) / (bowlers["balls"] / 6)).round(2)
top_bowlers = bowlers.sort_values("wickets", ascending=False).head(15)

fig = px.bar(
    top_bowlers,
    x="wickets",
    y="bowler",
    color="economy",
    orientation="h",
    title="Top IPL wicket takers in the included dataset",
    template="plotly_dark",
    color_continuous_scale="Turbo",
)
fig.update_layout(height=620, yaxis={"categoryorder": "total ascending"})
fig.show()

## Conclusions

1. IPL scoring has moved upward across the sample, with late-era seasons showing
   higher match totals and run rates than the early years.
2. Six-hitting is a major part of the scoring shift, not just incremental
   strike rotation.
3. Bowling economy rose, but wicket-taking remains a meaningful counterweight,
   suggesting adaptation rather than total bowler irrelevance.
4. Toss advantage is noisy and usually close to a coin flip; chase preference is
   often more visible than a true toss-win edge.

### Reproducibility Notes

- Run `python3 scripts/validate_data.py` before opening the notebook.
- Run `python3 scripts/create_notebook.py` to regenerate this notebook.
- Run `python3 scripts/process_data.py` only when the raw Cricsheet CSV archive
  is available under `data/ipl_raw/`.